In [1]:
import sys, logging

MODULE_BASE = "/mnt/c/Users/JatinJangid/.vscode/adtech-databricks/AdTech - Finance project/Adtech - generators/generator_src"
CONFIG_PATH = "/mnt/c/Users/JatinJangid/.vscode/adtech-databricks/AdTech - Finance project/Adtech - generators/generator_src/config/generator_config.yaml"

if MODULE_BASE not in sys.path:
    sys.path.insert(0, MODULE_BASE)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)

print(f"MODULE_BASE : {MODULE_BASE}")
print(f"CONFIG_PATH : {CONFIG_PATH}")
print("Environment ready")

MODULE_BASE : /mnt/c/Users/JatinJangid/.vscode/adtech-databricks/AdTech - Finance project/Adtech - generators/generator_src
CONFIG_PATH : /mnt/c/Users/JatinJangid/.vscode/adtech-databricks/AdTech - Finance project/Adtech - generators/generator_src/config/generator_config.yaml
Environment ready


In [2]:
from core.config_loader import ConfigLoader
from validation.schema_validator import SchemaValidator

# Override output path to point at Unity Catalog Volume
OVERRIDES = {
    "platform": {
        "output_base_path": "/mnt/c/Users/JatinJangid/.vscode/adtech-databricks/AdTech - Finance project/Adtech - generators/adtech-raw"
    }
}

config = ConfigLoader(CONFIG_PATH, overrides=OVERRIDES).load().config
validator = SchemaValidator(config, max_record_age_hours=72)

print(f"seed           : {config.platform.seed}")
print(f"user pool      : {config.users.total_pool_size}")
print(f"campaigns      : {config.campaigns.count}")
print(f"output path    : {config.platform.output_base_path}")
print(f"batch interval : {config.platform.batch_interval_seconds}s")
print("Config loaded")

01:20:01 [INFO] core.config_loader: Configuration loaded from /mnt/c/Users/JatinJangid/.vscode/adtech-databricks/AdTech - Finance project/Adtech - generators/generator_src/config/generator_config.yaml


seed           : 42
user pool      : 3000
campaigns      : 20
output path    : /mnt/c/Users/JatinJangid/.vscode/adtech-databricks/AdTech - Finance project/Adtech - generators/adtech-raw
batch interval : 5s
Config loaded


In [3]:
import os

VOLUME_PATH = "/mnt/c/Users/JatinJangid/.vscode/adtech-databricks/AdTech - Finance project/Adtech - generators/adtech-raw"

# Create sub-directories for each event type
EVENT_TYPES = [
    "bid_requests", "impressions", "clicks", "conversions",
    "campaign_cdc", "refunds", "billing_records", "ledger_entries"
]

for et in EVENT_TYPES:
    path = f"{VOLUME_PATH}/{et}"
    os.makedirs(path, exist_ok=True)

# Write and delete a test file to confirm write access
test_file = f"{VOLUME_PATH}/.write_test"
try:
    with open(test_file, "w") as f:
        f.write("ok")
    os.remove(test_file)
    print(f"✓ Volume is writable: {VOLUME_PATH}")
    print(f"✓ Event type dirs created: {EVENT_TYPES}")
except PermissionError:
    print("✗ PERMISSION DENIED on Volume.")
    print("  Fix: run the grant command below in a SQL cell:")
    print()
    print("  GRANT WRITE VOLUME ON VOLUME main.adtech_raw.`adtech-raw`")
    print("  TO `jatin.jangid.104@gmail.com`;")
except Exception as e:
    print(f"✗ Unexpected error: {e}")

✓ Volume is writable: /mnt/c/Users/JatinJangid/.vscode/adtech-databricks/AdTech - Finance project/Adtech - generators/adtech-raw
✓ Event type dirs created: ['bid_requests', 'impressions', 'clicks', 'conversions', 'campaign_cdc', 'refunds', 'billing_records', 'ledger_entries']


In [4]:
# Restart Python to pick up the fixed source files
dbutils.library.restartPython()

In [5]:
from orchestrator import GeneratorOrchestrator
import time

print("Initialising orchestrator (building user pool + campaigns)...")
t0 = time.time()
orchestrator = GeneratorOrchestrator(config, validator)
elapsed = time.time() - t0

print(f"Orchestrator ready in {elapsed:.1f}s")
print(f"Generators  : {[g.EVENT_TYPE for g in orchestrator._generators]}")
print(f"Output path : {config.platform.output_base_path}")

01:20:02 [INFO] simulation.shared_state: IP pools: 140 clean, 10 fraud


Initialising orchestrator (building user pool + campaigns)...


01:20:02 [INFO] simulation.shared_state: User pool initialized: 3000 users (120 bots)


01:20:02 [INFO] simulation.shared_state: Campaign pool initialized: 20 campaigns


01:20:02 [INFO] edge_cases.schema_evolution_injector: Schema evolution controller initialized at V1 for all event types


Orchestrator ready in 0.6s
Generators  : ['bid_requests', 'impressions', 'clicks', 'conversions', 'refunds', 'billing_records', 'ledger_entries', 'campaign_cdc']
Output path : /mnt/c/Users/JatinJangid/.vscode/adtech-databricks/AdTech - Finance project/Adtech - generators/adtech-raw


In [6]:
# For your first run, use max_iterations=20 to verify everything works.
# Change to None for continuous infinite streaming.

orchestrator.run(max_iterations=20)

01:20:02 [INFO] orchestrator: ======================================================================


01:20:02 [INFO] orchestrator: AdTech Data Generator V2 — All 14 FAANG Challenges Active


01:20:02 [INFO] orchestrator:   Active generators: ['bid_requests', 'impressions', 'clicks', 'conversions', 'refunds', 'billing_records', 'ledger_entries', 'campaign_cdc']


01:20:02 [INFO] orchestrator:   Schema evolution: True


01:20:02 [INFO] orchestrator:   Time chaos: True


...

01:21:42 [INFO] orchestrator:   Records:       10,867


01:21:42 [INFO] orchestrator:   Errors:        0


01:21:42 [INFO] orchestrator:   Throughput:    108.6 rps


01:21:42 [INFO] orchestrator:   Schema state:  {'bid_requests': 1, 'clicks': 1, 'conversions': 1}


01:21:42 [INFO] orchestrator: ======================================================================


In [7]:
import os

VOLUME_PATH = "/mnt/c/Users/JatinJangid/.vscode/adtech-databricks/AdTech - Finance project/Adtech - generators/adtech-raw"

print(f"{'Event Type':<25} {'Files':>6} {'Records':>10}")
print("─" * 45)

total_records = 0
for et in ["bid_requests","impressions","clicks","conversions",
           "campaign_cdc","refunds","billing_records","ledger_entries"]:
    path = f"{VOLUME_PATH}/{et}"
    if not os.path.exists(path):
        print(f"{et:<25} {'no dir':>6}")
        continue
    files, records = 0, 0
    for root, _, fs in os.walk(path):
        for fname in fs:
            if fname.endswith(".json"):
                files += 1
                try:
                    with open(os.path.join(root, fname)) as f:
                        records += sum(1 for _ in f)
                except Exception:
                    pass
    total_records += records
    print(f"{et:<25} {files:>6} {records:>10,}")

print("─" * 45)
print(f"{'TOTAL':<25} {'':>6} {total_records:>10,}")

Event Type                 Files    Records
─────────────────────────────────────────────


bid_requests                  20      6,965
impressions                   20      3,154


clicks                        20        316
conversions                   20         45
campaign_cdc                  20         24


refunds                       20         23
billing_records               20        182
ledger_entries                20        158
─────────────────────────────────────────────
TOTAL                                10,867
